# 0.AVISO

O objetivo desse projeto é poder praticar exercitar o conteúdo abordado no meu curso de pós-graduação, sobretudo, visão computacional. Portanto, investirei pouco tempo analisando as variáveis e focarei muito mais em entender a distribuição das imagens.

# 1.Setup

In [1]:
import pandas as pd
import polars as pl
import polars.selectors as cs
from scipy.stats import chi2_contingency, mannwhitneyu

import numpy as np

import altair as alt

from collections import Counter

from util import funcoes_exploracao as fe  # funcoes que criei

from sc_detect.config import RAW_DATA_DIR



2026-10-23 10:11:08.377 | INFO     | sc_detect.config:<module>:11 - PROJ_ROOT path is: /home/gabriel_santiago/Estudos/Portfolio/skin_cancer_detection


In [2]:
# definindo a seed
pl.set_random_seed(42)

In [3]:
# configuracoes altair
alt.renderers.enable("default")
alt.data_transformers.enable("vegafusion", max_rows=None)
alt.theme.enable("fivethirtyeight")

ThemeRegistry.enable('fivethirtyeight')

# 0.Pequeno Teste

Vou testar Polars ao invés de Pandas pela primeira vez, então, aproveitarei para fazer algumas comparações simples e rápidas.

In [4]:
train_metadados_pd = pd.read_csv(RAW_DATA_DIR / 'train-metadata.csv')

/tmp/ipykernel_7444/100571722.py:1: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  train_metadados_pd = pd.read_csv(RAW_DATA_DIR / 'train-metadata.csv')


In [5]:
train_metadados = pl.read_csv(RAW_DATA_DIR / 'train-metadata.csv')

In [6]:
train_metadados_pd.head()

,isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,...,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
0,ISIC_0015670,0,IP_1235828,60.0,male,lower extremity,3.04,TBP tile: close-up,3D: white,20.244422,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,97.517282
1,ISIC_0015845,0,IP_8170065,60.0,male,head/neck,1.10,TBP tile: close-up,3D: white,31.712570,...,IL_6727506,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,3.141455
2,ISIC_0015864,0,IP_6724798,60.0,male,posterior torso,3.40,TBP tile: close-up,3D: XP,22.575830,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.804040
3,ISIC_0015902,0,IP_4111386,65.0,male,anterior torso,3.22,TBP tile: close-up,3D: XP,14.242329,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.989998
4,ISIC_0024200,0,IP_8313778,55.0,male,anterior torso,2.73,TBP tile: close-up,3D: white,24.725520,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,70.442510


In [7]:
train_metadados.head()

isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,tbp_lv_Aext,tbp_lv_B,tbp_lv_Bext,tbp_lv_C,tbp_lv_Cext,tbp_lv_H,tbp_lv_Hext,tbp_lv_L,tbp_lv_Lext,tbp_lv_areaMM2,tbp_lv_area_perim_ratio,tbp_lv_color_std_mean,tbp_lv_deltaA,tbp_lv_deltaB,tbp_lv_deltaL,tbp_lv_deltaLB,tbp_lv_deltaLBnorm,tbp_lv_eccentricity,tbp_lv_location,tbp_lv_location_simple,tbp_lv_minorAxisMM,tbp_lv_nevi_confidence,tbp_lv_norm_border,tbp_lv_norm_color,tbp_lv_perimeterMM,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z,attribution,copyright_license,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
str,i64,str,str,str,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,f64
"""ISIC_0015670""",0,"""IP_1235828""","""60""","""male""","""lower extremity""",3.04,"""TBP tile: close-up""","""3D: white""",20.244422,16.261975,26.922447,23.954773,33.684638,28.953117,53.058545,55.828924,54.367448,62.025701,3.152561,27.47617,0.0,3.982447,2.967674,-7.658253,8.360566,5.784302,0.901302,"""Right Leg - Upper""","""Right Leg""",1.543016,0.002629,7.09136,0.0,9.307003,0.0,2.036195,2.63778,0.590476,85,-182.703552,613.493652,-42.427948,"""Memorial Sloan Kettering Cance…","""CC-BY""","""""","""Benign""","""Benign""","""""","""""","""""","""""","""""","""NA""",97.517282
"""ISIC_0015845""",0,"""IP_8170065""","""60""","""male""","""head/neck""",1.1,"""TBP tile: close-up""","""3D: white""",31.71257,25.36474,26.331,24.54929,41.21903,35.29926,39.70291,44.06404,48.86152,55.36236,0.9194971,12.23529,0.0,6.34783,1.781713,-6.500838,6.839008,4.987244,0.639885,"""Head & Neck""","""Head & Neck""",0.8219178,1.3343e-7,2.116402,0.0,3.354148,0.0,0.8532267,3.912844,0.2857143,55,-0.078308,1575.687,57.1745,"""Memorial Sloan Kettering Cance…","""CC-BY""","""IL_6727506""","""Benign""","""Benign""","""""","""""","""""","""""","""""","""NA""",3.141455
"""ISIC_0015864""",0,"""IP_6724798""","""60""","""male""","""posterior torso""",3.4,"""TBP tile: close-up""","""3D: XP""",22.57583,17.12817,37.97046,33.48541,44.17492,37.6118,59.26585,62.90973,53.96118,61.67052,3.265153,24.18462,0.0,5.447655,4.485044,-7.709336,9.092376,6.290359,0.932147,"""Torso Back Top Third""","""Torso Back""",1.194905,0.000296,4.798335,0.0,8.886309,0.0,1.743651,1.950777,0.3619048,105,123.6497,1472.01,232.9089,"""Memorial Sloan Kettering Cance…","""CC-BY""","""""","""Benign""","""Benign""","""""","""""","""""","""""","""""","""NA""",99.80404
"""ISIC_0015902""",0,"""IP_4111386""","""65""","""male""","""anterior torso""",3.22,"""TBP tile: close-up""","""3D: XP""",14.242329,12.164757,21.448144,21.121356,25.7462,24.374023,56.414429,60.060388,18.649518,23.314841,6.07994,14.889242,0.51452,2.077572,0.326788,-4.665323,4.783413,6.400196,0.654458,"""Torso Front Top Half""","""Torso Front""",2.481328,21.989453,1.975874,1.771705,9.514499,0.66469,1.258541,1.573733,0.209581,130,-141.02478,1442.185791,58.359802,"""ACEMID MIA""","""CC-0""","""""","""Benign""","""Benign""","""""","""""","""""","""""","""""","""NA""",99.989998
"""ISIC_0024200""",0,"""IP_8313778""","""55""","""male""","""anterior torso""",2.73,"""TBP tile: close-up""","""3D: white""",24.72552,20.05747,26.4649,25.71046,36.21798,32.60874,46.94607,52.04118,46.27631,54.85574,2.101708,19.90256,0.0,4.668053,0.7544338,-8.579431,9.148495,6.531302,0.9464478,"""Torso Front Top Half""","""Torso Front""",0.929916,0.001379,3.658854,0.0,6.467562,0.0,2.085409,2.480509,0.3134328,20,-72.31564,1488.72,21.42896,"""Memorial Sloan Kettering Cance…","""CC-BY""","""""","""Benign""","""Benign""","""""","""""","""""","""""","""""","""NA""",70.44251


In [8]:
tam_polars = train_metadados.estimated_size() / (1024 ** 2)
print(f"RAM consumida pelo DataFrame no Polars: {tam_polars:.2f} MB")

RAM consumida pelo DataFrame no Polars: 169.83 MB


In [9]:
tam_pandas = train_metadados_pd.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"RAM consumida pelo DataFrame no Pandas: {tam_pandas:.2f} MB")

RAM consumida pelo DataFrame no Pandas: 511.97 MB


Uma diferença um tanto quanto ignorante.

In [10]:
# limpando a memoria
del tam_pandas, train_metadados_pd

# 2.EDA

In [11]:
# vou testar o metodo lazy do polars nesse projeto
train_metadados = pl.scan_csv(
    RAW_DATA_DIR / 'train-metadata.csv',
    null_values = ["", "NA", "NAN", "N/A", "null", "None"])

## 2.1.Visão Geral

In [12]:
train_metadados.head().collect()

isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,tbp_lv_Aext,tbp_lv_B,tbp_lv_Bext,tbp_lv_C,tbp_lv_Cext,tbp_lv_H,tbp_lv_Hext,tbp_lv_L,tbp_lv_Lext,tbp_lv_areaMM2,tbp_lv_area_perim_ratio,tbp_lv_color_std_mean,tbp_lv_deltaA,tbp_lv_deltaB,tbp_lv_deltaL,tbp_lv_deltaLB,tbp_lv_deltaLBnorm,tbp_lv_eccentricity,tbp_lv_location,tbp_lv_location_simple,tbp_lv_minorAxisMM,tbp_lv_nevi_confidence,tbp_lv_norm_border,tbp_lv_norm_color,tbp_lv_perimeterMM,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z,attribution,copyright_license,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
str,i64,str,i64,str,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,f64
"""ISIC_0015670""",0,"""IP_1235828""",60,"""male""","""lower extremity""",3.04,"""TBP tile: close-up""","""3D: white""",20.244422,16.261975,26.922447,23.954773,33.684638,28.953117,53.058545,55.828924,54.367448,62.025701,3.152561,27.47617,0.0,3.982447,2.967674,-7.658253,8.360566,5.784302,0.901302,"""Right Leg - Upper""","""Right Leg""",1.543016,0.002629,7.09136,0.0,9.307003,0.0,2.036195,2.63778,0.590476,85,-182.703552,613.493652,-42.427948,"""Memorial Sloan Kettering Cance…","""CC-BY""",null,"""Benign""","""Benign""",null,null,null,null,null,null,97.517282
"""ISIC_0015845""",0,"""IP_8170065""",60,"""male""","""head/neck""",1.1,"""TBP tile: close-up""","""3D: white""",31.71257,25.36474,26.331,24.54929,41.21903,35.29926,39.70291,44.06404,48.86152,55.36236,0.9194971,12.23529,0.0,6.34783,1.781713,-6.500838,6.839008,4.987244,0.639885,"""Head & Neck""","""Head & Neck""",0.8219178,1.3343e-7,2.116402,0.0,3.354148,0.0,0.8532267,3.912844,0.2857143,55,-0.078308,1575.687,57.1745,"""Memorial Sloan Kettering Cance…","""CC-BY""","""IL_6727506""","""Benign""","""Benign""",null,null,null,null,null,null,3.141455
"""ISIC_0015864""",0,"""IP_6724798""",60,"""male""","""posterior torso""",3.4,"""TBP tile: close-up""","""3D: XP""",22.57583,17.12817,37.97046,33.48541,44.17492,37.6118,59.26585,62.90973,53.96118,61.67052,3.265153,24.18462,0.0,5.447655,4.485044,-7.709336,9.092376,6.290359,0.932147,"""Torso Back Top Third""","""Torso Back""",1.194905,0.000296,4.798335,0.0,8.886309,0.0,1.743651,1.950777,0.3619048,105,123.6497,1472.01,232.9089,"""Memorial Sloan Kettering Cance…","""CC-BY""",null,"""Benign""","""Benign""",null,null,null,null,null,null,99.80404
"""ISIC_0015902""",0,"""IP_4111386""",65,"""male""","""anterior torso""",3.22,"""TBP tile: close-up""","""3D: XP""",14.242329,12.164757,21.448144,21.121356,25.7462,24.374023,56.414429,60.060388,18.649518,23.314841,6.07994,14.889242,0.51452,2.077572,0.326788,-4.665323,4.783413,6.400196,0.654458,"""Torso Front Top Half""","""Torso Front""",2.481328,21.989453,1.975874,1.771705,9.514499,0.66469,1.258541,1.573733,0.209581,130,-141.02478,1442.185791,58.359802,"""ACEMID MIA""","""CC-0""",null,"""Benign""","""Benign""",null,null,null,null,null,null,99.989998
"""ISIC_0024200""",0,"""IP_8313778""",55,"""male""","""anterior torso""",2.73,"""TBP tile: close-up""","""3D: white""",24.72552,20.05747,26.4649,25.71046,36.21798,32.60874,46.94607,52.04118,46.27631,54.85574,2.101708,19.90256,0.0,4.668053,0.7544338,-8.579431,9.148495,6.531302,0.9464478,"""Torso Front Top Half""","""Torso Front""",0.929916,0.001379,3.658854,0.0,6.467562,0.0,2.085409,2.480509,0.3134328,20,-72.31564,1488.72,21.42896,"""Memorial Sloan Kettering Cance…","""CC-BY""",null,"""Benign""","""Benign""",null,null,null,null,null,null,70.44251


In [13]:
# qtd linhas do df
qtd_linhas = train_metadados.select(pl.len()).collect().item()
print(f'Número de linhas do dataframe: {qtd_linhas} linhas.')

Número de linhas do dataframe: 401059 linhas.


In [14]:
# contagem de tipos
print('Tipos dos dados: ')
Counter(list(train_metadados.collect_schema().values()))

Tipos dos dados: 


Counter({Float64: 33, String: 19, Int64: 3})

In [15]:
print('Distribuicao de nulos do Dataset:')
nulos = fe.pl_dist_nulos(train_metadados)
nulos

Distribuicao de nulos do Dataset:


variaveis,qtd_nulos,percent_nulos
str,u32,f64
"""iddx_5""",401058,100.0
"""mel_mitotic_index""",401006,99.99
"""mel_thick_mm""",400996,99.98
"""iddx_4""",400508,99.86
"""iddx_3""",399994,99.73
"""iddx_2""",399991,99.73
"""lesion_id""",379001,94.5
"""sex""",11517,2.87
"""anatom_site_general""",5756,1.44


Não irei me aprofundar na análise de nulos para esse projeto, portanto, apenas irei optar por excluir as colunas com mais de 40% dos valores nulos.

In [16]:
# extraindo colunas com muitos nulos
cols_null_drop = (
    nulos
        .filter(pl.col('percent_nulos') >= 40)
        .get_column('variaveis')
        .to_list()
)

print(cols_null_drop)

['iddx_5', 'mel_mitotic_index', 'mel_thick_mm', 'iddx_4', 'iddx_3', 'iddx_2', 'lesion_id']


In [17]:
# estatistica do df
train_metadados.describe()

statistic,isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,tbp_lv_Aext,tbp_lv_B,tbp_lv_Bext,tbp_lv_C,tbp_lv_Cext,tbp_lv_H,tbp_lv_Hext,tbp_lv_L,tbp_lv_Lext,tbp_lv_areaMM2,tbp_lv_area_perim_ratio,tbp_lv_color_std_mean,tbp_lv_deltaA,tbp_lv_deltaB,tbp_lv_deltaL,tbp_lv_deltaLB,tbp_lv_deltaLBnorm,tbp_lv_eccentricity,tbp_lv_location,tbp_lv_location_simple,tbp_lv_minorAxisMM,tbp_lv_nevi_confidence,tbp_lv_norm_border,tbp_lv_norm_color,tbp_lv_perimeterMM,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z,attribution,copyright_license,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
str,str,f64,str,f64,str,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,f64
"""count""","""401059""",401059.0,"""401059""",398261.0,"""389542""","""395303""",401059.0,"""401059""","""401059""",401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,"""401059""","""401059""",401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,"""401059""","""401059""","""22058""","""401059""","""401059""","""1068""","""1065""","""551""","""1""","""53""","""63""",401059.0
"""null_count""","""0""",0.0,"""0""",2798.0,"""11517""","""5756""",0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""379001""","""0""","""0""","""399991""","""399994""","""400508""","""401058""","""401006""","""400996""",0.0
"""mean""",null,0.00098,null,58.012986,null,null,3.930827,null,null,19.974007,14.919247,28.281706,26.913015,34.786341,30.921279,54.653689,60.996869,42.289976,51.199529,8.539975,19.084517,1.070408,5.05476,1.368691,-8.909552,9.455766,7.538922,0.741238,null,null,2.539773,38.520265,3.451523,3.09159,11.878891,1.016459,2.71519,2.238605,0.306823,86.332073,-3.091862,1039.598221,55.823389,null,null,null,null,null,null,null,null,null,null,null,97.162204
"""std""",null,0.031288,null,13.596165,null,null,1.743068,null,null,3.999489,3.529384,5.278676,4.482994,5.708469,4.829345,5.520849,5.631909,10.864572,12.100527,9.679312,5.357863,0.762918,2.642928,2.215871,3.463362,3.466703,2.404951,0.143857,null,null,1.173169,41.480936,1.724557,2.044813,5.919302,0.734631,1.738165,0.623884,0.125038,52.559511,197.257995,409.819653,87.968245,null,null,null,null,null,null,null,null,null,null,null,8.995782
"""min""","""ISIC_0015670""",0.0,"""IP_0008821""",5.0,"""female""","""anterior torso""",1.0,"""TBP tile: close-up""","""3D: XP""",-2.487115,-9.080269,-0.730989,9.237066,3.054228,11.84652,-1.574164,28.43649,4.223007,17.70226,0.431601,10.761634,0.0,-13.81985,-18.088106,-38.75614,2.952675,3.001138,0.027667,"""Head & Neck""","""Head & Neck""",0.273973,0.0,0.5894259,0.0,2.579237,0.0,0.26816,0.636247,0.052034,0.0,-624.870728,-1052.134,-291.890442,"""ACEMID MIA""","""CC-0""","""IL_0001200""","""Benign""","""Benign""","""Benign adnexal epithelial prol…","""Angiofibroma""","""Actinic keratosis, Bowenoid""","""Blue nevus, Cellular""","""0/mm^2""","""0.2""",1.2611e-16
"""25%""",null,0.0,null,50.0,null,null,2.84,null,null,17.330821,12.46974,24.70438,23.84813,31.003187,27.65829,51.566295,57.29765,34.30339,42.27451,4.109589,15.426205,0.574909,3.413109,0.1110402,-10.67545,6.952381,5.748166,0.6566313,null,null,1.812707,0.10982,2.14384,1.728782,8.338364,0.5638921,1.456571,1.834745,0.2114286,40.0,-147.021851,746.521545,-8.96228,null,null,null,null,null,null,null,null,null,null,null,99.66883
"""50%""",null,0.0,null,60.0,null,null,3.37,null,null,19.80191,14.71393,28.17157,26.701704,3

## 2.2.Análises Univariadas

In [18]:
# checando dist da coluna target
fe.pl_contagem_valores(train_metadados, 'target', excluir_na=False).collect()

target,qtd_target,percent_target
i64,u32,f64
0,400666,99.9
1,393,0.1


Temos um problema extremamente desbalanceado.

In [19]:
# checando distribuicao da coluna sex
fe.pl_contagem_valores(train_metadados, 'sex', excluir_na=False).collect()


sex,qtd_sex,percent_sex
str,u32,f64
null,11517,2.87
"""male""",265546,66.21
"""female""",123996,30.92


In [20]:
# describe da idade
df_idade_train = train_metadados.select(pl.col('age_approx'))
df_idade_train.describe()

statistic,age_approx
str,f64
"""count""",398261.0
"""null_count""",2798.0
"""mean""",58.012986
"""std""",13.596165
"""min""",5.0
"""25%""",50.0
"""50%""",60.0
"""75%""",70.0
"""max""",85.0


Temos um coeficiente de variação (std / mean) de aproximadamente 23.5%, o que indica uma alta variabilidade nos dados. Irei plotar um histograma para compreender melhor essa variável, mas espero uma curva a esquerda.

In [21]:
# plotagem
(
    alt.Chart(df_idade_train.collect())
    .mark_bar()
    .encode(
        x=alt.X("age_approx", title="Idade").bin(maxbins=20),
        y=alt.Y("count()", title="Quantidade"),
    )
    .properties(width=600, height=400, title="Distribuição de Idade")
)

alt.Chart(...)

Nossa base de dados está concentrada principalmente no público entre 50 e 70 anos, o que pode enviesar nosso modelo. Ao cruzar essa variável com a target, precisarei verificar seu poder preditivo afim de pensar em como tratar esse problema.

In [22]:
train_metadados.select(pl.all().n_unique()).collect()

isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,tbp_lv_Aext,tbp_lv_B,tbp_lv_Bext,tbp_lv_C,tbp_lv_Cext,tbp_lv_H,tbp_lv_Hext,tbp_lv_L,tbp_lv_Lext,tbp_lv_areaMM2,tbp_lv_area_perim_ratio,tbp_lv_color_std_mean,tbp_lv_deltaA,tbp_lv_deltaB,tbp_lv_deltaL,tbp_lv_deltaLB,tbp_lv_deltaLBnorm,tbp_lv_eccentricity,tbp_lv_location,tbp_lv_location_simple,tbp_lv_minorAxisMM,tbp_lv_nevi_confidence,tbp_lv_norm_border,tbp_lv_norm_color,tbp_lv_perimeterMM,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z,attribution,copyright_license,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
401059,2,1042,17,3,6,1758,1,2,386052,385304,389890,387763,390703,388865,389798,390743,395726,396358,8029,167648,371189,398257,398886,396505,395511,396877,396648,21,8,75669,395242,391372,368742,9492,367566,396459,390080,74825,36,398446,382410,392817,7,3,22059,52,3,15,26,27,2,8,20,131480


In [23]:
valores_unicos = (
    train_metadados
        .select(pl.all().n_unique())
        .unpivot(variable_name='variaveis', value_name='qtd_valores_unicos')
        .sort('qtd_valores_unicos', descending=False)
    ).collect()
valores_unicos

variaveis,qtd_valores_unicos
str,u32
"""image_type""",1
"""target""",2
"""tbp_tile_type""",2
"""iddx_5""",2
"""sex""",3
…,…
"""tbp_lv_deltaLBnorm""",396877
"""tbp_lv_deltaA""",398257
"""tbp_lv_x""",398446


image_type também pode ser dropada.

In [24]:
train_metadados.select(pl.col('image_type')).unique().collect()

image_type
str
"""TBP tile: close-up"""


In [25]:
train_metadados.select(pl.col('tbp_tile_type')).unique().collect()

tbp_tile_type
str
"""3D: XP"""
"""3D: white"""


In [26]:
train_metadados.select(pl.col('copyright_license')).unique().collect()

copyright_license
str
"""CC-BY-NC"""
"""CC-BY"""
"""CC-0"""


In [27]:
train_metadados.select(pl.col('attribution')).unique().collect()

attribution
str
"""Department of Dermatology, Hos…"
"""Memorial Sloan Kettering Cance…"
"""Department of Dermatology, Uni…"
"""ViDIR Group, Department of Der…"
"""ACEMID MIA"""
"""University Hospital of Basel"""
"""Frazer Institute, The Universi…"


As variáveis: tbp_lv_x, tbp_lv_y, tbp_lv_z fornecem a posição exata da lesão e isso depende muito da estatura do paciente. 

Para inserir a localização na aprendizagem do modelo, irei trabalhar com outras variáveis, as categóricas: anatom_site_general, tbp_lv_location ou tbp_lv_location_simple.

## 2.3.Data Cleaning

In [28]:
## DROPANDO COLUNAS NULAS E NAO PREDITIVAS
# selecionando as colunas a serem dropadas
columns_to_drop = [
    'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z',  # vars de posicao  (nao preditivas tambem)
    'attribution', 'copyright_license', 'image_type',  # nao preditivas
    # 'anatom_site_general', 'sex'
      
] + cols_null_drop # nulas

# dropando
df_clean = train_metadados.drop(columns_to_drop)
print(f'Colunas dropadas: {columns_to_drop}')

Colunas dropadas: ['tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z', 'attribution', 'copyright_license', 'image_type', 'iddx_5', 'mel_mitotic_index', 'mel_thick_mm', 'iddx_4', 'iddx_3', 'iddx_2', 'lesion_id']


In [29]:
# calculando quantos registros de target 1 
# perderemos em caso de exclusao de alguns nulos

# calculando qtd de target 1 no dataset de treino
col_nul = ['sex', 'anatom_site_general', 'age_approx']
qtd_target_1_geral = fe.pl_contagem_valores(train_metadados, 'target').filter(pl.col('target') == 1).select('qtd_target').collect().item()

# calculo de perda
for c in col_nul:
    aux = df_clean.filter(~pl.col(c).is_null())
    qtd_target_1_filtrado = (
        fe.pl_contagem_valores(aux, 'target')  # calculando qtd de target 1
            .filter(pl.col('target') == 1)
            .select('qtd_target')
            .collect()
            .item()
        )
    perdidos = qtd_target_1_geral - qtd_target_1_filtrado
    prop_perdidos = perdidos / qtd_target_1_geral
    print(f'Excluindo nulos de {c} perderemos {perdidos} ({prop_perdidos:.2%}) registros de target = 1.')

Excluindo nulos de sex perderemos 10 (2.54%) registros de target = 1.
Excluindo nulos de anatom_site_general perderemos 0 (0.00%) registros de target = 1.
Excluindo nulos de age_approx perderemos 3 (0.76%) registros de target = 1.


Por se tratar de um problema desbalanceado, irei preservar ao máximo a quantidade de registros de target = 1. Portanto, irei preencher sex com a moda e idade com a mediana.

Irei dropar os registros sem anatom_site_general pois pode ser perigoso inferir o local da lesão tanto para o aprendizado do modelo quanto para o paciente.

Para simplificar, usarei apenas sexo, idade e local da lesão para auxiliar na predição.

## 2.3.Analise Bivariada

Quero testar se as variáveis de idade, sexo e local da lesão têm influência direta na target. Para isso, usarei testes estatísticos.

In [30]:
# amostrando para analises
mdt_sample = train_metadados.collect().sample(fraction=0.25)
qtd_linhas = mdt_sample.select(pl.col('target')).count().item()
print(f'Dataframe de treino amostrado em 25% resultou em: {qtd_linhas} linhas.')

Dataframe de treino amostrado em 25% resultou em: 100264 linhas.


### 2.3.1.Variável 'Sex'

In [31]:
# visualizacao da variavel de sexo x target
for t in [0, 1]:
    df_aux = mdt_sample.filter(pl.col('target') == t)
    graf = (
        alt.Chart(df_aux)
        .mark_bar()
        .encode(
            x=alt.X("sex", title="Sexo"),
            y=alt.Y("count()", title="Quantidade"),
        )
        .properties(width=400, height=200, title=f"Distribuição da Target por Sexo = {t}")
    )
    graf.show()


alt.Chart(...)

alt.Chart(...)

A distribuição da target parece ter se mantido entre os sexos, supunho que essa variável não têm relação forte o suficiente com a target. Mas, vamos ao teste.

Para testar essa hipótese, por se tratar de duas variáveis categóricas, escolhi o teste Qui-Quadrado:

* Hipótese Nula (H0): O sexo do paciente e a incidência do tumor (target) são independentes.
* Hipótese Alternativa (H1): O sexo do paciente e a incidência do tumor são dependentes.

In [45]:
# gerando tabela de contingencia
df_contingencia = (
    train_metadados
    .filter(pl.col('sex').is_not_null())
    .group_by("sex")
    .agg([
        (pl.col("target") == 0).sum().alias("benigno"),
        (pl.col("target") == 1).sum().alias("maligno")
    ])
    .sort("sex")
).collect()

# plotando tabela
print("Tabela de Contingência:")
print(df_contingencia)

# executando teste
matriz_frequencias = df_contingencia.select(["benigno", "maligno"]).to_numpy()  # extraindo valores para o teste
chi2, p_valor, dof, esperados = chi2_contingency(matriz_frequencias)

print("\n--- Resultado do Teste Estatístico ---")
print(f"Estatística Chi2: {chi2:.4f}")
print(f"p-valor: {p_valor:.4e}")
print(f"Graus de Liberdade: {dof}")

# interpretacao dos resultados
alpha = 0.05
if p_valor < alpha:
    print("\nConclusão: Rejeitamos H0. Existe uma associação estatisticamente significativa entre 'sex' e 'target'.")
else:
    print("\nConclusão: Falhamos em rejeitar H0. Não há evidência estatística de que 'sex' influencie a 'target'.")

Tabela de Contingência:
shape: (2, 3)
┌────────┬─────────┬─────────┐
│ sex    ┆ benigno ┆ maligno │
│ ---    ┆ ---     ┆ ---     │
│ str    ┆ u32     ┆ u32     │
╞════════╪═════════╪═════════╡
│ female ┆ 123887  ┆ 109     │
│ male   ┆ 265272  ┆ 274     │
└────────┴─────────┴─────────┘

--- Resultado do Teste Estatístico ---
Estatística Chi2: 1.8560
p-valor: 1.7308e-01
Graus de Liberdade: 1

Conclusão: Falhamos em rejeitar H0. Não há evidência estatística de que 'sex' influencie a 'target'.


### 2.3.2.Variável 'age_approx'

In [33]:
# visualizacao da variavel de idade x target
for t in [0, 1]:
    df_aux = mdt_sample.filter(pl.col('target') == t)
    graf = (
        alt.Chart(df_aux)
        .mark_bar()
        .encode(
            x=alt.X("age_approx", title="Idade").bin(maxbins=20),
            y=alt.Y("count()", title="Quantidade"),
        )
        .properties(width=400, height=200, title=f"Distribuição de Idade com Target = {t}")
    )
    graf.show()


alt.Chart(...)

alt.Chart(...)

Já nesse caso, suspeito de que possa haver influência de idade na target.

Para esse teste, irei usar o teste não paramétrico Teste U:
* Hipótese Nula (H0): As distribuições da idade para lesões benignas e malignas são iguais. A idade não tem impacto na probabilidade de malignidade.

* Hipótese Alternativa (H1): As distribuições da idade do grupo com lesões malignas são significativamente diferentes do grupo com lesões benignas.

In [47]:
# agrupando
df_idades = (
    train_metadados
    .filter(pl.col("age_approx").is_not_null())
    .group_by("target")
    .agg(pl.col("age_approx").implode().alias("idades"))
).collect()

# extraindo as dist
idades_benigno = df_idades.filter(pl.col("target") == 0)["idades"].item().to_numpy()
idades_maligno = df_idades.filter(pl.col("target") == 1)["idades"].item().to_numpy()

# executando o teste
stat_u, p_valor_u = mannwhitneyu(idades_maligno, idades_benigno, alternative='two-sided')

# resultados
print(f"Média Idade (Benigno): {idades_benigno.mean():.2f} anos")
print(f"Média Idade (Maligno): {idades_maligno.mean():.2f} anos")
print("-" * 50)
print(f"[Mann-Whitney U]  Estatística: {stat_u:.4f} | p-valor: {p_valor_u:.4e}")

# interpretacao dos resultados
alpha = 0.05
if p_valor < alpha:
    print("\nConclusão: Rejeitamos H0. Existe uma associação estatisticamente significativa entre 'age_approx' e 'target'.")
else:
    print("\nConclusão: Falhamos em rejeitar H0. Não há evidência estatística de que 'age_approx' influencie a 'target'.")

Média Idade (Benigno): 58.01 anos
Média Idade (Maligno): 61.37 anos
--------------------------------------------------
[Mann-Whitney U]  Estatística: 88535852.5000 | p-valor: 1.1996e-06

Conclusão: Rejeitamos H0. Existe uma associação estatisticamente significativa entre 'age_approx' e 'target'.


### 2.3.3.Variável 'anatom_site_general'

In [41]:
# visualizacao da variavel de idade x target
for t in [0, 1]:
    df_aux = mdt_sample.filter(pl.col('target') == t)
    graf = (
        alt.Chart(df_aux)
        .mark_bar()
        .encode(
            x=alt.X("anatom_site_general", title="Local da Lesão"),
            y=alt.Y("count()", title="Quantidade"),
        )
        .properties(width=400, height=200, title=f"Distribuição da Target = {t} com Local da Lesão")
    )
    graf.show()


alt.Chart(...)

alt.Chart(...)

Vejo fortes indícios de que o local da lesão influencia fortemente no diagnóstico.

Para esse teste também irei usar Qui-Quadrado:
* Hipótese Nula (H0): O local anatômico da lesão (anatom_site_general) e a taxa de malignidade (target) são independentes.
* Hipótese Alternativa (H1): A probabilidade da lesão ser maligna varia de acordo com a região do corpo.

In [46]:
# gerando tabela de contingencia
df_contingencia = (
    train_metadados
    .filter(pl.col("anatom_site_general").is_not_null())
    .group_by("anatom_site_general")
    .agg([
        (pl.col("target") == 0).sum().alias("benigno"),
        (pl.col("target") == 1).sum().alias("maligno")
    ])
    .sort("anatom_site_general")
).collect()

# plotando tabela
print("Tabela de Contingência:")
print(df_contingencia)

# executando teste
matriz_frequencias = df_contingencia.select(["benigno", "maligno"]).to_numpy()  # extraindo valores para o teste
chi2, p_valor, dof, esperados = chi2_contingency(matriz_frequencias)

print("\n--- Resultado do Teste Estatístico ---")
print(f"Estatística Chi2: {chi2:.4f}")
print(f"p-valor: {p_valor:.4e}")
print(f"Graus de Liberdade: {dof}")

# interpretacao dos resultados
alpha = 0.05
if p_valor < alpha:
    print("\nConclusão: Rejeitamos H0. Existe uma associação estatisticamente significativa entre 'anatom_site_general' e 'target'.")
else:
    print("\nConclusão: Falhamos em rejeitar H0. Não há evidência estatística de que 'anatom_site_general' influencie a 'target'.")

Tabela de Contingência:
shape: (5, 3)
┌─────────────────────┬─────────┬─────────┐
│ anatom_site_general ┆ benigno ┆ maligno │
│ ---                 ┆ ---     ┆ ---     │
│ str                 ┆ u32     ┆ u32     │
╞═════════════════════╪═════════╪═════════╡
│ anterior torso      ┆ 87688   ┆ 82      │
│ head/neck           ┆ 11968   ┆ 78      │
│ lower extremity     ┆ 102955  ┆ 73      │
│ posterior torso     ┆ 121799  ┆ 103     │
│ upper extremity     ┆ 70500   ┆ 57      │
└─────────────────────┴─────────┴─────────┘

--- Resultado do Teste Estatístico ---
Estatística Chi2: 378.3416
p-valor: 1.3284e-80
Graus de Liberdade: 4

Conclusão: Rejeitamos H0. Existe uma associação estatisticamente significativa entre 'anatom_site_general' e 'target'.
